# पाठ 13 - कॉगनी नॉलेज ग्राफ़्स के साथ एजेंट मेमोरी


## सेटअप

यह नोटबुक दिखाता है कि कैसे [**Cognee**](https://www.cognee.ai/) नॉलेज ग्राफ़ और **Microsoft Agent Framework** (MAF) का उपयोग करके स्थायी मेमोरी के साथ एक बुद्धिमान **कोडिंग सहायक** बनाया जाए।

Cognee असंरचित टेक्स्ट को संरचित, क्वेरी योग्य नॉलेज ग्राफ़ में बदल देता है जो वेक्टर एम्बेडिंग्स द्वारा समर्थित होता है — जो आपके एजेंट को एक समृद्ध, संबंध-जागरूक दीर्घकालिक मेमोरी प्रदान करता है।

### आप क्या सीखेंगे
1. **नॉलेज ग्राफ़ बनाना**: डेवलपर प्रोफाइल और सर्वोत्तम प्रथाओं को संरचित, क्वेरी योग्य ज्ञान में बदलना।
2. **Cognee को MAF के साथ इंटीग्रेट करना**: MAF एजेंट को Cognee के नॉलेज ग्राफ़ को क्वेरी करने के लिए `@tool` फंक्शंस का उपयोग करना।
3. **सेशन-जागरूक संवाद**: एक ही सेशन में कई प्रश्नों के बीच संदर्भ बनाए रखना।
4. **दीर्घकालिक मेमोरी**: महत्वपूर्ण ज्ञान को सेशनों के बीच बनाए रखना और नए संवादों में पुनः प्राप्त करना।

### आवश्यकताएँ
- Python 3.9+
- लोकल में Redis चलाना (`docker run -d -p 6379:6379 redis`) सेशन प्रबंधन के लिए
- LLM API key (जैसे OpenAI) — `.env` में `LLM_API_KEY` सेट करें
- `.env` में `CACHING=true` (Cognee सेशनों के लिए आवश्यक)
- Azure AI Foundry प्रोजेक्ट जिसमें एक तैनात चैट मॉडल हो
- `.env` में `AZURE_AI_PROJECT_ENDPOINT` और `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेंट मेमोरी के प्रकार

यह नोटबुक मुख्य पाठ 13 नोटबुक के समान तीन मेमोरी प्रकारों का अन्वेषण करती है, लेकिन लांग-टर्म मेमोरी बैकएंड के रूप में Cognee का उपयोग करती है:

| मेमोरी प्रकार | तंत्र | जीवनकाल |
|---|---|---|
| **वर्किंग** | `agent.create_session()` (MAF) | एकल संवाद |
| **शॉर्ट-टर्म** | Cognee सेशन कैश (Redis) | एकल सत्र |
| **लॉन्ग-टर्म** | Cognee ज्ञान ग्राफ + वेक्टर | स्थायी |

### Cognee की मेमोरी संरचना
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कॉग्नी स्टोरेज तैयार करें


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग 1 — नॉलेज बेस बनाना

हम अपने कोडिंग असिस्टेंट के लिए एक व्यापक नॉलेज बेस बनाने के लिए तीन प्रकार के डेटा का उपयोग करते हैं:

1. **डेवलपर प्रोफ़ाइल** — व्यक्तिगत विशेषज्ञता और तकनीकी पृष्ठभूमि  
2. **पाइथन बेहतरीन प्रथाएँ** — पाइथन का ज़ेन साथ ही व्यावहारिक दिशानिर्देश  
3. **ऐतिहासिक वार्तालाप** — डेवलपर्स और AI असिस्टेंट के बीच पिछले प्रश्नोत्तर सत्र


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफ़ का विज़ुअलाइज़ेशन करें

Cognee उन संस्थाओं और संबंधों का एक इंटरैक्टिव HTML विज़ुअलाइज़ेशन प्रस्तुत कर सकता है जिन्हें उसने निकाला है।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमोरी को मेमिफाई के साथ समृद्ध करें

`memify()` ज्ञान ग्राफ का विश्लेषण करता है और बुद्धिमान नियम उत्पन्न करता है — पैटर्न, सर्वोत्तम प्रथाओं, और अवधारणाओं के बीच संबंधों की पहचान करता है।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग 2 — Cognee उपकरणों के साथ MAF एजेंट

अब हम एक MAF एजेंट बनाएंगे जो `@tool` फ़ंक्शंस के माध्यम से Cognee के नॉलेज ग्राफ से प्रश्न कर सके। यह एजेंट को ग्राफ-आधारित सेमांटिक सर्च की पूरी शक्ति का लाभ उठाने देता है, जबकि सत्रों के माध्यम से संवादात्मक संदर्भ बनाए रखता है।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## सत्रों के साथ कार्य स्मृति

`AgentSession` (जो `agent.create_session()` के माध्यम से बनाया जाता है) सत्र के भीतर कार्य स्मृति प्रदान करता है। एजेंट पिछले संदेशों का संदर्भ ले सकता है और साथ ही Cognee के दीर्घकालिक नॉलेज ग्राफ से प्रश्न पूछ सकता है।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नया सत्र — दीर्घकालिक स्मृति बनी रहती है

एक नया सत्र शुरू करने से वर्किंग मेमोरी साफ़ हो जाती है, लेकिन Cognee नॉलेज ग्राफ़ अभी भी उपलब्ध है। एजेंट एक बिल्कुल नए संवाद में वही दीर्घकालिक ज्ञान पुनः प्राप्त कर सकता है।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

इस नोटबुक में आपने एक कोडिंग सहायक बनाया जो **MAF की कार्यशील स्मृति** (`agent.create_session()`) को **Cognee के दीर्घकालिक ज्ञान ग्राफ** के साथ जोड़ता है।

### आपने क्या सीखा
1. **ज्ञान ग्राफ निर्माण**: Cognee बिना संरचित पाठ को ग्रहण करता है और एक ग्राफ + वेक्टर मेमोरी बनाता है।
2. **memify के साथ ग्राफ समृद्धि**: आपके मौजूदा ग्राफ पर व्युत्पन्न तथ्यों और अधिक समृद्ध संबंधों को जोड़ा गया।
3. **MAF + Cognee एकीकरण**: `@tool` फ़ंक्शन MAF एजेंट को Cognee के ग्राफ को स्वाभाविक रूप से क्वेरी करने देते हैं।
4. **कार्यशील स्मृति + दीर्घकालिक स्मृति**: `AgentSession` (जो `agent.create_session()` से आता है) सत्र संदर्भ प्रदान करता है जबकि Cognee स्थायी ज्ञान प्रदान करता है।
5. **NodeSets के साथ फ़िल्टर्ड सर्च**: ज्ञान ग्राफ के विशिष्ट उपसमूहों को लक्षित करें (जैसे केवल सिद्धांत)।

### मुख्य बातें
- **Cognee** कच्चे पाठ को संरचित, संबंध-ज्ञानी स्मृति में बदलता है — जो एक सपाट वेक्टर स्टोर की तुलना में अधिक शक्तिशाली है।
- **`@tool` फ़ंक्शन** MAF एजेंट्स और बाहरी ज्ञान प्रणालियों के बीच सहज पुल का काम करते हैं।
- **`AgentSession`** (जो `agent.create_session()` से आता है) प्रति-वार्तालाप संदर्भ को दीर्घकालिक ज्ञान से अलग रखता है।
- एक ही ज्ञान ग्राफ कई सत्रों और एजेंट्स की सेवा करता है।

### वास्तविक दुनिया के अनुप्रयोग
- **डेवलपर कपिलॉट्स**: कोड समीक्षा, घटना विश्लेषण, वास्तुकला सहायक
- **ग्राहक-सामना करने वाले कपिलॉट्स**: उत्पाद दस्तावेज़, अक्सर पूछे जाने वाले प्रश्न, और CRM नोट्स पर सपोर्ट एजेंट्स
- **आंतरिक विशेषज्ञ कपिलॉट्स**: दिशानिर्देशों के ऊपर नीति, कानूनी, या सुरक्षा सहायक तर्कशीलता
- **एकीकृत डेटा परतें**: संरचित और असंरचित डेटा को एक क्वेरी योग्य ग्राफ में संयोजित करें

### अगले कदम
- Cognee में कालानुक्रमिक जागरूकता के साथ प्रयोग करें
- डोमेन-विशिष्ट ग्राफ गुणवत्ता के लिए OWL ऑन्तोलॉजी परिभाषित करें
- समय के साथ पुनः प्राप्ति में सुधार के लिए उपयोगकर्ता प्रतिक्रिया लूप जोड़ें
- एक ही Cognee मेमोरी परत साझा करने वाली बहु-एजेंट प्रणालियों के लिए पैमाना बढ़ाएं


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
इस दस्तावेज़ का अनुवाद एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवाद में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
